## Import Libraries

In [1]:
import glob, os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## Load and Combine CSV Files

In [2]:
brand_map = {"merc": "Mercedes", "cclass": "Mercedes", "focus": "Ford",
             "hyundi": "Hyundai", "vw": "Volkswagen"}

frames = []
for f in glob.glob("data/*.csv"):
    name = os.path.basename(f).replace(".csv", "").replace("unclean ", "").lower()
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={"tax(£)": "tax", "fuel type": "fuelType", "engine size": "engineSize"})
    df["brand"] = brand_map.get(name, name.capitalize())
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
data.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,brand
0,A1,2017,12500,Manual,15735,Petrol,150.0,55.4,1.4,Audi
1,A6,2016,16500,Automatic,36203,Diesel,20.0,64.2,2.0,Audi
2,A1,2016,11000,Manual,29946,Petrol,30.0,55.4,1.4,Audi
3,A4,2017,16800,Automatic,25952,Diesel,145.0,67.3,2.0,Audi
4,A3,2019,17300,Manual,1998,Petrol,145.0,49.6,1.0,Audi


## Basic Cleaning

In [3]:
cols = ["brand", "year", "price", "transmission", "mileage", "fuelType", "tax", "mpg", "engineSize"]
data = data.reindex(columns=cols)

for c in ["year", "price", "mileage", "tax", "mpg", "engineSize"]:
    data[c] = pd.to_numeric(data[c], errors="coerce")

data = data.dropna().drop_duplicates()
data = data[(data["engineSize"] > 0) & (data["year"].between(1990, 2026))]

print(data.shape)
data["brand"].value_counts()

(97438, 9)


brand
Ford          17759
Volkswagen    14878
Vauxhall      13224
Mercedes      12848
Bmw           10614
Audi          10512
Toyota         6693
Skoda          6183
Hyundai        4727
Name: count, dtype: int64

## Create Car Age

In [4]:
data["age"] = 2026 - data["year"]
data.to_csv("cars_clean.csv", index=False)
data.head()

,brand,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,age
0,Audi,2017,12500,Manual,15735,Petrol,150.0,55.4,1.4,9
1,Audi,2016,16500,Automatic,36203,Diesel,20.0,64.2,2.0,10
2,Audi,2016,11000,Manual,29946,Petrol,30.0,55.4,1.4,10
3,Audi,2017,16800,Automatic,25952,Diesel,145.0,67.3,2.0,9
4,Audi,2019,17300,Manual,1998,Petrol,145.0,49.6,1.0,7


## Encode Categorical Features

In [5]:
X = data[["brand", "age", "mileage", "transmission", "fuelType", "tax", "mpg", "engineSize"]]
X = pd.get_dummies(X)
y = data["price"]
X.shape

(97438, 23)

## Train Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Train Models

In [7]:
models = {
    "Decision Tree": DecisionTreeRegressor(max_depth=12, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)

## Compare Models

In [8]:
rows = []
for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
        "MAE": mean_absolute_error(y_test, test_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

,Train R2,Test R2,MAE,RMSE
Model,,,,
Decision Tree,0.947,0.911,1712.047,2983.058
Random Forest,0.987,0.944,1305.896,2363.967
XGBoost,0.960,0.942,1407.371,2399.207


## Save Best Model

In [9]:
best_name = results["Test R2"].idxmax()
print("Best model:", best_name)
joblib.dump({"model": models[best_name], "columns": list(X.columns)}, "price_model.pkl", compress=3)

Best model: Random Forest


['price_model.pkl']